# ConQRR (2020)
[[paper]](https://arxiv.org/pdf/2012.11867)<br>
ConQRR = **Con**versational **Q**uery **R**ewriting for **R**etrieval

ConQRR — это модель для переписывания поисковых запросов в контексте диалога, разработанная исследователями из Amazon. Она преобразует неполные или контекстно-зависимые реплики пользователя в полноценные, семантически завершенные запросы, которые могут быть эффективно обработаны стандартными системами Retrieval.

### Задача
В Conversational Search пользователи часто используют краткие фразы, содержащие Coreference (местоимения «он», «это») или Ellipsis (пропуск слов, понятных из контекста). Например:
- *User:* «Расскажи про Сатурн».
- *User:* «А сколько у **него** спутников?»
Стандартный Dense Retrieval не сможет найти ответ на второй вопрос, так как в нем нет ключевого слова «Сатурн». Задача ConQRR — переписать второй запрос в «Сколько спутников у Сатурна?».

### Существующие решения
До появления ConQRR задачу решали следующими методами:
- Coreference Resolution модели: специализированные системы (например, на базе компонентов из SpaCy или AllenNLP), которые заменяют местоимения на сущности. Недостаток: не умеют восстанавливать контекст при полном пропуске слов (Ellipsis).
- Supervised Query Rewriting (2019): использование Sequence-to-Sequence моделей (GPT-2, T5), обученных на парах «контекст + запрос -> эталонный переписанный запрос». Недостаток: обучение через Cross-Entropy минимизирует разницу в тексте, но не гарантирует, что полученный запрос будет лучше работать в поиске (проблема несоответствия метрик генерации и метрик поиска).

### Идея
Авторы предложили не просто обучать модель генерировать текст, похожий на человеческий, а использовать Reinforcement Learning (RL), чтобы оптимизировать модель напрямую под метрики качества поиска (Recall, MRR). Это позволяет модели выучить, какие именно слова важны для поискового движка, даже если итоговая фраза выглядит менее «литературной».

### Архитектура
ConQRR строится на базе Transformer-архитектуры Sequence-to-Sequence:
- **Encoder**: принимает на вход склеенную последовательность из истории диалога ($H$) и текущего запроса ($q_n$).
- **Decoder**: генерирует переписанный запрос ($q^*_n$).
В качестве базовой модели обычно используется **T5** (2019), так как она предобучена на задачах преобразования текста в текст.

### Алгоритм обучения
Процесс разбит на два этапа:

1.  **Supervised Pre-training**:
    - Модель обучается на датасетах типа CANARD (наборы данных для переписывания запросов).
    - Используется стандартный **Maximum Likelihood Estimation (MLE)** для предсказания следующего токена. Это дает модели базовое понимание грамматики и контекста.

2.  **Reinforcement Learning Fine-tuning**:
    - Модель рассматривается как **Agent**, а генерация запроса — как последовательность действий (**Actions**).
    - Для каждого сгенерированного запроса выполняется реальный поиск (например, через BM25 или DPR) по большой коллекции документов.
    - Вычисляется **Reward function**: используется метрика **Recall@N** или **MRR** (Mean Reciprocal Rank) результатов поиска.
    - Обновление весов происходит с помощью алгоритма **Policy Gradient** (в частности, Self-Critical Sequence Training). Если сгенерированный запрос привел к выдаче нужного документа на первом месте, веса, ответственные за генерацию этих токенов, подкрепляются.

### Алгоритм инференса
1.  На вход подается текущая реплика пользователя и $k$ предыдущих реплик диалога.
2.  Модель выполняет **Beam Search** или **Greedy Decoding** для генерации одной строки (переписанного запроса).
3.  Полученный запрос отправляется в стандартный поисковый индекс (Elasticsearch или векторную базу).

### Результаты
- На датасете **CANARD** использование ConQRR с RL-дообучением увеличило метрику **Recall@10** на 6-8 процентных пунктов по сравнению с базовой моделью T5, обученной только через MLE.
- Авторы доказали, что прямая оптимизация под поиск (RL) работает лучше, чем простое увеличение размера модели: ConQRR на базе T5-small показывает результаты, сопоставимые с T5-base без RL-блока, при этом работая значительно быстрее.
- Модель показала устойчивость к «шумному» контексту: за счет RL она научилась игнорировать нерелевантные части диалога, которые при обычном обучении часто попадали в запрос и портили выдачу.

## 📝 Критический анализ

```markdown
# ConQRR (2020)
---
[[paper]](https://arxiv.org/pdf/2012.11867)<br>
ConQRR = **Con**versational **Q**uery **R**ewriting for **R**etrieval

ConQRR — модель для переписывания запросов в диалогах, разработанная Amazon. Она преобразует неполные реплики в полноценные запросы для эффективной обработки Retrieval-системами.

### Задача
В Conversational Search пользователи часто используют краткие фразы с Coreference или Ellipsis. Например:
- *User:* «Расскажи про Сатурн».
- *User:* «А сколько у **него** спутников?»
ConQRR переписывает второй запрос в «Сколько спутников у Сатурна?».

### Альтернативы
До ConQRR использовались:
- **Coreference Resolution**: заменяет местоимения на сущности, но не восстанавливает контекст при Ellipsis.
- **Supervised Query Rewriting** (2019): Sequence-to-Sequence модели (GPT-2, T5), обученные на парах «контекст + запрос -> переписанный запрос», но не оптимизированные под метрики поиска.

### Идея
Использование Reinforcement Learning (RL) для оптимизации под метрики поиска (Recall, MRR), что позволяет модели выделять важные для поиска слова.

### Архитектура
ConQRR основана на Transformer Sequence-to-Sequence:
- **Encoder**: принимает историю диалога и текущий запрос.
- **Decoder**: генерирует переписанный запрос.
Базовая модель — **T5** (2019).

### Алгоритм обучения
1. **Supervised Pre-training**:
   - Обучение на датасетах типа CANARD с использованием **Maximum Likelihood Estimation (MLE)**.

2. **Reinforcement Learning Fine-tuning**:
   - Модель как **Agent**, генерация запроса — **Actions**.
   - Реальный поиск по коллекции документов.
   - **Reward function**: метрика **Recall@N** или **MRR**.
   - Обновление весов через **Policy Gradient**.

### Алгоритм инференса
1. Ввод текущей реплики и $k$ предыдущих реплик.
2. **Beam Search** или **Greedy Decoding** для генерации запроса.
3. Запрос отправляется в поисковый индекс.

### Результаты
- На датасете **CANARD** RL-дообучение увеличило **Recall@10** на 6-8 п.п. по сравнению с T5, обученной только через MLE.
- ConQRR на базе T5-small сопоставима с T5-base без RL, но работает быстрее.
- Модель устойчива к «шумному» контексту, игнорируя нерелевантные части диалога.

<img src="img/img.png" width=500>
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример иллюстрации основных концепций модели ConQRR (2020) для переписывания запросов в диалоговых системах.

# Импортируем необходимые библиотеки
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# Инициализируем токенизатор и модель T5
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

# Пример истории диалога и текущего запроса
dialogue_history = "User: Расскажи про Сатурн. User: А сколько у него спутников?"
current_query = "А сколько у него спутников?"

# Конкатенируем историю диалога и текущий запрос
input_text = f"{dialogue_history} {current_query}"

# Токенизируем входной текст
input_ids = tokenizer.encode(input_text, return_tensors='pt')

# Генерируем переписанный запрос с помощью модели
# Используем beam search для более качественного результата
outputs = model.generate(input_ids, max_length=50, num_beams=5, early_stopping=True)

# Декодируем сгенерированные токены в текст
rewritten_query = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Выводим переписанный запрос
print("Переписанный запрос:", rewritten_query)

# Пример использования Reinforcement Learning для дообучения модели
# (упрощенный псевдокод, так как RL требует сложной инфраструктуры для обучения)

# Определяем функцию награды (например, на основе Recall@N)
def reward_function(rewritten_query, ground_truth):
    # Здесь мы сравниваем сгенерированный запрос с эталонным
    # и возвращаем награду на основе метрики поиска
    return recall_at_n(rewritten_query, ground_truth)

# Псевдокод для RL-дообучения
def reinforcement_learning_fine_tuning(model, data_loader):
    for batch in data_loader:
        # Получаем входные данные и эталонные переписанные запросы
        input_ids, ground_truth = batch
        
        # Генерируем переписанные запросы
        outputs = model.generate(input_ids)
        
        # Вычисляем награду для каждого сгенерированного запроса
        rewards = [reward_function(output, gt) for output, gt in zip(outputs, ground_truth)]
        
        # Обновляем веса модели с помощью Policy Gradient
        # (например, с использованием Self-Critical Sequence Training)
        model.update_weights(outputs, rewards)

# Пример инференса с использованием переписанного запроса
# (упрощенный пример, без реального поиска)
def perform_search(rewritten_query):
    # Здесь мы бы отправили переписанный запрос в поисковую систему
    # и получили бы результаты
    search_results = ["Документ 1", "Документ 2", "Документ 3"]
    return search_results

# Выполняем поиск с переписанным запросом
search_results = perform_search(rewritten_query)
print("Результаты поиска:", search_results)
```

### Комментарии к коду:
1. **Инициализация модели и токенизатора**: Используем T5-small, так как он подходит для задач преобразования текста в текст.
2. **Конкатенация истории диалога и текущего запроса**: Это позволяет модели учитывать контекст при генерации переписанного запроса.
3. **Генерация переписанного запроса**: Используем beam search для получения более качественного результата.
4. **Reinforcement Learning (RL) дообучение**: Псевдокод показывает, как можно использовать RL для оптимизации модели под метрики поиска, такие как Recall@N.
5. **Инференс и поиск**: Переписанный запрос отправляется в поисковую систему для получения результатов.